# Week 5, Day 3: Ask the WWE store through LangChain

**Lesson:** [week5/day3.ipynb](../../../week5/day3.ipynb), *Chroma, a retriever and an LLM*. **About 30 minutes.**

**What you practice here**

- The two LangChain objects of a RAG pipeline: a retriever from the Chroma store you built, reopened with the same embeddings, and `ChatOpenAI` at temperature 0.
- A system prompt template with a `{context}` slot, filled with `.format`.
- The RAG answer: `retriever.invoke` for the documents, their `page_content` joined together, then `llm.invoke` with a `SystemMessage` and a `HumanMessage`, and the reply read from `.content`.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week5/day3", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson reopens the Chroma store it built on Day 2, sets up its two key LangChain objects, a retriever and an LLM,
and answers a question in five lines: retrieve, join, fill the template, invoke, read the content. Do the same for
the store of your WWE documents that you built in Day 2's Try it for real.

`make_retriever` and `make_llm` take LangChain's `Chroma` and `ChatOpenAI` classes as their first parameter, and
`answer_question` takes the retriever and the LLM they give. That lets the checks pass stand-ins shaped like them,
which record what your code sends and answer with canned replies, so nothing is spent. `answer_question` builds its
messages with LangChain's `SystemMessage` and `HumanMessage`, imported in the imports cell as the lesson does; the
checks read the messages it sends by their type and content.

## What each call should give

```python
retriever = make_retriever(Chroma, embeddings)
# the retriever of the Chroma store saved in "vector_db", opened with these embeddings

llm = make_llm(ChatOpenAI)
# a ChatOpenAI chat model for gpt-4.1-nano, at temperature 0

SYSTEM_PROMPT_TEMPLATE.format(context="Hacksaw Jim Duggan won the first Royal Rumble match.")
# '...questions about the history of WWE...Context:\nHacksaw Jim Duggan won the first Royal Rumble match.\n'

answer_question(retriever, llm, "Who won the first Royal Rumble match?")
# retrieves the documents for the question, invokes the model once with a system message (the template filled with
# their text) and a human message (the question), and returns the reply's text:
# 'Hacksaw Jim Duggan won the first Royal Rumble match, in 1988.'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
from langchain_core.messages import SystemMessage, HumanMessage

### 1 · `make_retriever(Chroma, embeddings, db_name="vector_db")`

Opens the vector store you built and returns its retriever. Open the Chroma store saved in the folder `db_name`, with
`embeddings` as its embedding function, as the lesson does, and return the retriever the store gives. `embeddings`
must be the same model that built the store, or the question's vector will not match the chunks' vectors. Pass
LangChain's `Chroma` class for real; the checks pass a stand-in.

<details><summary>Hint</summary>

```python
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
return vectorstore.as_retriever()
```

</details>

In [ ]:
def make_retriever(Chroma, embeddings, db_name="vector_db"):
    # Write make_retriever here, as described above.
    raise NotImplementedError("make_retriever is not written yet")

In [ ]:
check("make_retriever")

### 2 · `make_llm(ChatOpenAI, model="gpt-4.1-nano", temperature=0)`

Builds the chat model, as the lesson does: a `ChatOpenAI` with the given model name and temperature, both passed by
name, and returns it. Temperature 0 always picks the most likely next token, so the answers stay predictable.

<details><summary>Hint</summary>

```python
return ChatOpenAI(temperature=temperature, model_name=model)
```

LangChain also takes `model=` in place of `model_name=`, and so does the check.

</details>

In [ ]:
def make_llm(ChatOpenAI, model="gpt-4.1-nano", temperature=0):
    # Write make_llm here, as described above.
    raise NotImplementedError("make_llm is not written yet")

In [ ]:
check("make_llm")

### 3 · `SYSTEM_PROMPT_TEMPLATE`

The system prompt, with a slot that `answer_question` fills with the retrieved documents. Write it as a string with
exactly one slot, named `context`, and no other braces, so that the string's `format` method can fill it. Like the
lesson's, it says the assistant answers questions about the history of WWE from the given context, and what to do
when it does not know. The check looks for "WWE", the word "know" and the one slot.

<details><summary>Hint</summary>

```python
SYSTEM_PROMPT_TEMPLATE = """
You are ... (who the assistant is, and that it should use the context)
If you don't know the answer, say so.
Context:
{context}
"""
```

</details>

In [ ]:
SYSTEM_PROMPT_TEMPLATE = ""

In [ ]:
check("SYSTEM_PROMPT_TEMPLATE")

### 4 · `answer_question(retriever, llm, question)`

Answers one question with RAG, as the lesson's `answer_question` does, and returns the reply's text. Ask the
retriever for the documents for `question`, once, and join their text with a blank line between them. Fill the
template's slot with that context. Then invoke the model once with two messages, a `SystemMessage` holding the filled
template and a `HumanMessage` holding the question, and return the content of the reply it gives.

<details><summary>Hint</summary>

```python
docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)
system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
return response.content
```

</details>

In [ ]:
def answer_question(retriever, llm, question):
    # Write answer_question here, as described above.
    raise NotImplementedError("answer_question is not written yet")

In [ ]:
check("answer_question")

## Check everything

The cell below runs all 7 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 5, Day 3 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

This needs the `vector_db` folder that Day 2's Try it for real built at the top of your fork, and the same embedding model. Each answer from `gpt-4.1-nano` costs a fraction of a cent.

1. Load your keys with `load_dotenv(override=True)`.
2. Make the same embeddings as on Day 2, `HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")`, then
   `retriever = make_retriever(Chroma, embeddings, db_name=str(ROOT / "vector_db"))` (the setup cell set `ROOT` to
   your fork's folder) and `llm = make_llm(ChatOpenAI)`.
3. Ask "Who won the first Royal Rumble match?": print each retrieved document's source and first words, then the
   answer of `llm.invoke(question).content`, with no retrieval, then `answer_question(retriever, llm, question)`.
4. Temperature, the lesson's sidebar: ask `llm` and `make_llm(ChatOpenAI, temperature=1)` the same playful question
   three times each, such as a ring name for a cowboy wrestler, and compare how much the answers vary. Then launch
   the chat: `gr.ChatInterface` with a function of `message` and `history` that calls `answer_question`, and
   `type="messages"`, then `.launch()`.

<details><summary>Stuck? One way to write it</summary>

```python
import gradio as gr
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # the model that built the store on Day 2
retriever = make_retriever(Chroma, embeddings, db_name=str(ROOT / "vector_db"))
llm = make_llm(ChatOpenAI)

question = "Who won the first Royal Rumble match?"
for doc in retriever.invoke(question):
    print(doc.metadata["source"], "|", doc.page_content[:60].replace("\n", " "))
print("Without retrieval:", llm.invoke(question).content)
print("With retrieval:", answer_question(retriever, llm, question))

hot = make_llm(ChatOpenAI, temperature=1)
for model in (llm, hot):
    print([model.invoke("Invent a ring name for a cowboy wrestler. Reply with the name only.").content
           for _ in range(3)])

gr.ChatInterface(lambda message, history: answer_question(retriever, llm, message), type="messages").launch()
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.